# Assignment: Tool‑Augmented LLM App (Weather + Calculator)

**Goal:** Build a minimal GenAI pipeline that uses **memory/caching** and **two tools** (weather + calculator), with a simple agent that routes to tools when needed.



## Instructions

- Implement the required functions in the **solution cells**
- Use a module‑level cache created by `setup_cache()`; repeated `(city, unit)` queries should hit the cache (no re‑fetch).
- Keep your code deterministic for grading.
- You may integrate LangChain locally, but the autograder focuses on tool behavior + caching and a minimal agent router.
- **Do not rename** required functions or change their signatures.

### Required functions

1. `setup_cache() -> Any`
2. `get_weather(city: str, unit: str = "C", fetcher: callable | None = None) -> dict`
3. `multiply(a: float, b: float) -> float`
4. `tool_registry() -> dict`
5. `agent_route(prompt: str) -> str`  (return `"weather"`, `"multiply"`, or `"none"`)
6. `agent_answer(prompt: str, tools: dict | None = None, cache=None) -> str`

### Playtests (not graded)
Try these prompts after you finish:
- “What’s the weather in Paris?”
- “What’s 42 × 19?”
- “Summarize both results in one sentence.”


- The required deterministic tool functions are implemented in the solution cells.
- `setup_cache()` creates the required dictionary cache and enables LangChain `InMemoryCache` when LangChain is installed.
- Weather results use Open-Meteo; no separate weather API key is needed.
- OpenRouter is configured optionally for future LLM-based extensions; the public/hidden grading tests focus on the tool functions, routing, and caching.


In [ ]:

# Read-Only imports (you may import more inside solutions)
from typing import Any, Callable, Dict, Optional
import re
import requests  # allowed for tests


## OpenRouter API Setup

This assignment is configured to use **OpenRouter** instead of provider-specific API keys. Enter your OpenRouter API key when prompted. Do not hard-code the key or save it in the notebook.

The starter assignment and grading logic focus on the required tool, caching, and routing functions. The OpenRouter configuration is provided so any optional LLM calls can use the same key.

In [ ]:
# OpenRouter configuration (optional for the deterministic weather + calculator tools).
# The required assignment functions use Open-Meteo and Python arithmetic.
# Add OPENROUTER_API_KEY to Colab Secrets if you later extend this notebook with an LLM.
import os
from getpass import getpass

try:
    from google.colab import userdata
    _or_key = userdata.get("OPENROUTER_API_KEY")
except Exception:
    _or_key = os.environ.get("OPENROUTER_API_KEY")

if _or_key:
    os.environ["OPENROUTER_API_KEY"] = _or_key
    print("OpenRouter API key detected (value hidden).")
else:
    print("OpenRouter key not required for the core graded tool workflow.")


### Q1. Implement `setup_cache()` *(Solution cell)*
Create a cache object (e.g., dict) and assign it to module-level `CACHE`. Return it.


### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [ ]:
CACHE: Any = None

def setup_cache() -> Any:
    """Initialize and assign a dictionary cache to module-level CACHE.

    Also enables LangChain's in-memory LLM cache when LangChain is installed.
    The dictionary is used for deterministic weather-result caching and grading.
    """
    global CACHE
    CACHE = {}
    try:
        # Optional LangChain cache for chains/models used in extensions.
        from langchain_core.globals import set_llm_cache
        from langchain_core.caches import InMemoryCache
        set_llm_cache(InMemoryCache())
    except ImportError:
        # The required tool application still works without optional LangChain packages.
        pass
    return CACHE

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `setup_cache`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# BEGIN PUBLIC TESTS
c = setup_cache()
assert c is not None and isinstance(c, dict)
assert 'CACHE' in globals() and isinstance(CACHE, dict)
# END PUBLIC TESTS


### Q2. Implement `get_weather(city, unit="C")` *(Solution cell)*
- Use Open-Meteo Geocoding to resolve city → (lat, lon).
- Call Open-Meteo Current Weather for temperature (Celsius).
- If `unit == "F"`, convert Celsius to Fahrenheit.
- Cache result dict as `{"city": city, "temp": <float>, "unit": unit}` under normalized key.
- Subsequent calls with same `(city, unit)` must return the **same dict object** from cache (so `is` equality works).


### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [ ]:
def get_weather(city: str, unit: str = "C", fetcher: Optional[Callable] = None) -> dict:
    """Return current weather for a city, caching by normalized (city, unit).

    Uses Open-Meteo's free geocoding and forecast APIs (no weather API key).
    `fetcher` is an optional injectable callable for tests; it should accept
    the city and return either Celsius temperature or a weather dictionary.
    """
    global CACHE
    if CACHE is None:
        setup_cache()
    if not isinstance(city, str) or not city.strip():
        raise ValueError("city must be a non-empty string")

    normalized_city = " ".join(city.strip().split())
    normalized_unit = str(unit).strip().upper()
    if normalized_unit in ("CELSIUS", "°C"):
        normalized_unit = "C"
    elif normalized_unit in ("FAHRENHEIT", "°F"):
        normalized_unit = "F"
    if normalized_unit not in ("C", "F"):
        raise ValueError("unit must be 'C' or 'F'")

    key = (normalized_city.casefold(), normalized_unit)
    cache_store = CACHE
    if key in cache_store:
        return cache_store[key]

    if fetcher is not None:
        try:
            fetched = fetcher(normalized_city)
        except TypeError:
            fetched = fetcher(city=normalized_city)
        if isinstance(fetched, dict):
            raw_temp = fetched.get("temp", fetched.get("temperature"))
            source_unit = str(fetched.get("unit", "C")).upper()
        else:
            raw_temp, source_unit = fetched, "C"
        if raw_temp is None:
            raise ValueError("fetcher did not return a temperature")
        celsius = float(raw_temp)
        if source_unit == "F":
            celsius = (celsius - 32.0) * 5.0 / 9.0
    else:
        geocode_url = "https://geocoding-api.open-meteo.com/v1/search"
        geo_response = requests.get(
            geocode_url,
            params={"name": normalized_city, "count": 1, "language": "en", "format": "json"},
            timeout=20
        )
        geo_response.raise_for_status()
        locations = geo_response.json().get("results") or []
        if not locations:
            raise ValueError(f"Could not find city: {normalized_city}")
        location = locations[0]
        weather_response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": location["latitude"],
                "longitude": location["longitude"],
                "current": "temperature_2m",
                "temperature_unit": "celsius"
            },
            timeout=20
        )
        weather_response.raise_for_status()
        weather_data = weather_response.json()
        current = weather_data.get("current") or {}
        raw_temp = current.get("temperature_2m")
        if raw_temp is None:
            raise ValueError("Weather service did not return a current temperature")
        celsius = float(raw_temp)

    temperature = celsius if normalized_unit == "C" else (celsius * 9.0 / 5.0) + 32.0
    result = {"city": normalized_city, "temp": float(temperature), "unit": normalized_unit}
    cache_store[key] = result
    return result

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `get_weather`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q2 — Public Tests (API call allowed)
# BEGIN PUBLIC TESTS
setup_cache()
d1 = get_weather("Paris", "C")
d2 = get_weather("paris", "c")
assert d1 is d2, "Cached object should be returned for repeated queries."
assert isinstance(d1["temp"], float) and d1["unit"] == "C"
# Fahrenheit conversion
d3 = get_weather("Paris", "F")
assert d3["unit"] == "F" and isinstance(d3["temp"], float)
# END PUBLIC TESTS


### Q3. Implement `multiply(a, b)` *(Solution cell)*
Return the product of `a` and `b`.


### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [ ]:
def multiply(a: float, b: float) -> float:
    """Multiply two numeric values and return their product."""
    return float(a) * float(b)

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `multiply`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q3 — Public Tests
# BEGIN PUBLIC TESTS
assert multiply(6, 7) == 42
assert abs(multiply(1.5, 2.0) - 3.0) < 1e-9
# END PUBLIC TESTS

In [ ]:
# Q3 — Hidden Tests


### Q4. Implement `tool_registry()` *(Solution cell)*
Return a dict mapping `{"weather": get_weather, "multiply": multiply}`.


### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [ ]:
def tool_registry() -> Dict[str, Callable]:
    """Expose the application's two tools by stable names."""
    return {"weather": get_weather, "multiply": multiply}

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `tool_registry`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q4 — Public Tests
# BEGIN PUBLIC TESTS
reg = tool_registry()
assert isinstance(reg, dict)
assert "weather" in reg and "multiply" in reg
assert callable(reg["weather"]) and callable(reg["multiply"])
# END PUBLIC TESTS

In [ ]:
# Q4 — Hidden Tests


### Q5. Implement `agent_route(prompt)` *(Solution cell)*
Return `"weather"`, `"multiply"`, or `"none"` based on the prompt.


### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [ ]:
def agent_route(prompt: str) -> str:
    """Choose a tool from the user's natural-language request."""
    if not isinstance(prompt, str):
        return "none"
    text = prompt.casefold()

    # Weather intent should be checked before generic arithmetic indicators.
    weather_terms = ("weather", "temperature", "forecast", "how hot", "how cold")
    if any(term in text for term in weather_terms):
        return "weather"

    # Recognize explicit multiplication symbols/words and common phrasing.
    if re.search(r"\b(multiply|multiplication|product|times)\b", text):
        return "multiply"
    if re.search(r"\d\s*(?:\*|×|x)\s*-?\d", text):
        return "multiply"
    return "none"

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `agent_route`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q5 — Public Tests
# BEGIN PUBLIC TESTS
assert agent_route("what's the weather in Jaipur?") == "weather"
assert agent_route("compute 42 x 19 please") == "multiply"
assert agent_route("say hello") == "none"
# END PUBLIC TESTS


In [ ]:
# Q5 — Hidden Tests


### Q6. Implement `agent_answer(prompt, tools=None, cache=None)` *(Solution cell)*
- Use `agent_route` to choose a tool.
- If weather: parse city from `"weather in <City>"`. Default unit `"C"`, but if prompt mentions "Fahrenheit" or "F", use `"F"`.
- If multiply: parse two numbers from `a x b` or `a * b` and compute result with `multiply`.
- Return a short string response. You may pass `cache` in to override the module cache.


### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [ ]:
def agent_answer(prompt: str, tools: dict | None = None, cache=None) -> str:
    """Route a prompt to a tool, execute it, and return a concise response.

    For unsupported/general prompts, returns a clear message. The optional
    cache argument lets callers provide a cache dictionary for weather calls.
    """
    if not isinstance(prompt, str) or not prompt.strip():
        return "Please enter a question."

    registry = tools if tools is not None else tool_registry()
    route = agent_route(prompt)
    if route == "weather":
        # Parse a city after 'in'; trim trailing unit/time/punctuation phrases.
        match = re.search(r"\bweather\s+(?:like\s+)?in\s+(.+?)(?:\?|\.|!|,|\s+in\s+(?:fahrenheit|celsius|[fc])\b|\s+(?:fahrenheit|celsius)\b|\s+[fc]\s*$)", prompt, re.I)
        if not match:
            match = re.search(r"\btemperature\s+(?:in|for)\s+(.+?)(?:\?|\.|!|,|\s+(?:fahrenheit|celsius)\b)", prompt, re.I)
        if not match:
            return "Please specify a city, for example: What's the weather in Paris?"
        city = match.group(1).strip(" ,.?!")
        unit = "F" if re.search(r"fahrenheit|\bF\b|°F", prompt, re.I) else "C"
        weather_tool = registry.get("weather")
        if not callable(weather_tool):
            return "Weather tool is not available."
        try:
            # Cache override is supported while retaining the module-level default.
            if cache is not None:
                global CACHE
                old_cache = CACHE
                CACHE = cache
                try:
                    result = weather_tool(city, unit)
                finally:
                    CACHE = old_cache
            else:
                result = weather_tool(city, unit)
            return f"Current weather in {result['city']}: {result['temp']:.1f}°{result['unit']}."
        except Exception as exc:
            return f"Unable to retrieve weather for {city} in {unit}: {exc}"

    if route == "multiply":
        match = re.search(r"(-?\d+(?:\.\d+)?)\s*(?:\*|×|\bx\b|\bby\b)\s*(-?\d+(?:\.\d+)?)", prompt, re.I)
        if not match:
            numbers = re.findall(r"-?\d+(?:\.\d+)?", prompt)
            if len(numbers) >= 2 and re.search(r"\b(multiply|product|times)\b", prompt, re.I):
                match_values = numbers[:2]
            else:
                return "Please provide two numbers to multiply, for example: 12 * 8."
        else:
            match_values = match.groups()
        a, b = float(match_values[0]), float(match_values[1])
        multiply_tool = registry.get("multiply")
        if not callable(multiply_tool):
            return "Calculator tool is not available."
        product = float(multiply_tool(a, b))
        return f"{a:g} * {b:g} = {product:g}"

    # A small deterministic fallback for the playtest's follow-up summary.
    if "summarize" in prompt.casefold() or "both results" in prompt.casefold():
        values = list((cache if isinstance(cache, dict) else (CACHE or {})).values())
        if values:
            weather_items = [v for v in values if isinstance(v, dict) and "temp" in v]
            if weather_items:
                latest = weather_items[-1]
                return f"The cached weather result is {latest['city']} at {latest['temp']:.1f}°{latest['unit']}; ask a calculation question to add a calculator result."
        return "I can summarize the weather and calculation results after both tools have been run in this session."

    return "I can help with current weather in a city or multiply two numbers. What would you like to do?"

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `agent_answer`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:

# Q6 — Public Tests
# BEGIN PUBLIC TESTS
cache = setup_cache()
tools = tool_registry()

o1 = agent_answer("What's the weather in Mumbai in Fahrenheit?", tools=tools, cache=cache)
assert "Mumbai" in o1 and ("F" in o1)

o2 = agent_answer("Please do 12 * 8", tools=tools, cache=cache)
assert "12.0 * 8.0 = 96.0" in o2 or "12 * 8 = 96" in o2
# END PUBLIC TESTS


In [ ]:

# Q6 — Hidden Tests


## Playtest log and comparison (complete after running)

Run the following cell to generate three example tool-augmented outputs. Then compare them with vanilla LLM responses if you choose to add an LLM call. The core graded notebook does not require an LLM completion for weather or arithmetic: the external weather API and deterministic calculator are the tools being evaluated.

In [ ]:
# Three required playtest prompts
playtest_prompts = [
    "What's the weather in Paris?",
    "What's 42 x 19?",
    "Summarize both results in one sentence."
]
playtest_cache = setup_cache()
playtest_tools = tool_registry()

for test_prompt in playtest_prompts:
    print(f"USER: {test_prompt}")
    print(f"TOOL-AUGMENTED: {agent_answer(test_prompt, tools=playtest_tools, cache=playtest_cache)}")
    print("-" * 60)


## Reflection (500–700 words)

**Personalize this reflection after running the notebook and reviewing your outputs.** Add one or two specific observations from your own tests rather than claiming results you did not observe.

This assignment helped me understand how a tool-augmented language-model application differs from a chatbot that only generates text. A standard language model can explain concepts and produce natural-language responses, but it does not automatically have access to live weather information or a reliable calculator. In this application, the system identifies the intent of a prompt and routes it to one of two tools: a weather function or a multiplication function. The weather tool uses the Open-Meteo geocoding service to convert a city name into geographic coordinates and then requests the current temperature. The calculator tool performs multiplication directly in Python. This separation makes the result more dependable because the model application delegates tasks to software designed for those operations instead of asking a language model to guess or calculate from memory.

The weather and calculator tools extend the application's capabilities in different ways. Weather is an external-data tool: it obtains information that can change over time and that is not reliably available from a model's training data. The calculator is a deterministic computation tool: for valid numeric inputs, it returns the mathematical product without the arithmetic errors that can occur in generated text. The tool registry exposes both functions under stable names, while the routing function classifies a user's request as weather, multiplication, or neither. The agent-answer function then extracts the relevant arguments, calls the selected tool, and formats the result for the user. This is a simple form of orchestration because multiple components work together to complete a request.

Caching improves efficiency by storing weather results under a normalized city-and-unit key. Normalization means that requests such as “Paris” and “paris” with the same temperature unit can reuse the same result. The cache returns the same dictionary object for repeated requests, so the application avoids another geocoding and weather API call. This can reduce response time, network traffic, and the number of requests sent to an external service. The cache is held in memory, however, so it is cleared when the notebook runtime restarts. Also, weather is time-sensitive: a cached result may become less current as time passes. A production system should therefore use a time-to-live (TTL) policy so that old weather data is refreshed after a defined interval.

There are trade-offs between simplicity, speed, complexity, and transparency. The local dictionary cache is easy to understand and requires no separate database, but it is not shared between users or persistent across sessions. The weather tool depends on network connectivity and the availability of the Open-Meteo service. The routing logic is transparent and deterministic, which makes it straightforward to test, but keyword and pattern matching is less flexible than a full LLM-based agent when users phrase requests in unexpected ways. A more advanced implementation could use structured function calling so the model selects a tool and supplies validated arguments. That approach is more flexible but introduces additional model latency, API cost, and opportunities for incorrect tool selection.

I would extend the application with hybrid caching that combines an in-memory cache for fast repeated requests with a persistent cache for reuse across sessions. I would add a TTL to weather entries and potentially cache other API responses according to their freshness requirements. More tools could be registered, such as currency conversion, news search, or a database lookup. For complex requests, function chaining could call multiple tools in sequence and pass the outputs into a final summarization step. For example, the system could retrieve the weather for two cities, calculate a temperature difference, and summarize the comparison. Finally, logging tool selection, arguments, response time, and errors would make the application easier to debug and audit. Overall, this assignment demonstrated that tools provide access to live data and precise operations, while memory and caching help make repeated interactions more efficient.

---

## Submission

Submit your completed notebook. Ensure it runs top‑to‑bottom without modification in a clean environment **with valid API keys set**. If an API key is required for optional LLM functionality, use the OpenRouter API key configured above. The required grading functions focus on tool behavior, caching, and agent routing.


---

<h2><center> Completing your test </center></h2>

- Once you have completed your test and answered all the questions:
  
- You can open the guide using the button located on the far right of your screen.

- The button will look like this image:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%204/additional_files/End_guide_button.png?raw=1)

- Clicking on this button will allow you to mark the test as complete using the button shown below:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%204/additional_files/completion_button.png?raw=1)

- Please click on **Mark as Completed** to end and submit your test.

- Thank you!
